# TerraEyes DynamicEarthNet semantic change detection
Train the 49-way land-cover transition model on a Colab GPU, keeping the archive and outputs in Google Drive.

Open this notebook in your Google account and run its cells. Select a GPU runtime (Runtime → Change runtime type → T4 GPU or better). A reset restarts data preparation; training can resume from the Drive checkpoint after preparation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys, json

DRIVE_ROOT = Path('/content/drive/MyDrive/TerraEyes')
ARCHIVE = DRIVE_ROOT / 'data/dynamicearthnet/dynamicearthnet-video-71psnr.tacozip'
PROJECT_ROOT = Path('/content/AKASH_PRAHARI')
OUTPUT_ROOT = DRIVE_ROOT / 'training_outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_URL = 'https://github.com/Itzdvbravo/AKASH_PRAHARI.git'
REPO_REF = 'main'  # Set to a branch or commit containing the semantic training scripts
print('Archive:', ARCHIVE, 'exists:', ARCHIVE.is_file())

## Put the dataset archive in Drive
Copy `data/dynamicearthnet/dynamicearthnet-video-71psnr.tacozip` (about 2.2 GB) to `MyDrive/TerraEyes/data/dynamicearthnet/`. The split CSV comes from the repository. Do not commit or upload the archive to GitHub.

In [ ]:
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', 'FETCH_HEAD'], check=True)
assert ARCHIVE.is_file(), f'Copy the DynamicEarthNet archive to {ARCHIVE}'
DATA_ROOT = PROJECT_ROOT / 'data/dynamicearthnet'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print('Repository revision:', subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', '--short', 'HEAD'], text=True).strip())

## Install runtime dependencies and check GPU
Colab supplies PyTorch and CUDA. Install the dataset reader and FFmpeg needed to decode the TACO video archive.

In [ ]:
%cd /content/AKASH_PRAHARI
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install tacoreader==2.4.13 h5py
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Runtime → Change runtime type, then rerun this cell.')

## Train
AOIs are split before sampling. The script writes the best checkpoint to Drive during training. `--resume-from` continues after a Colab reconnect once data preparation has run again. Defaults sample 8 monthly pairs and 8 tiles per pair per AOI; adjust epochs for a longer run.

In [ ]:
CHECKPOINT = OUTPUT_ROOT / 'mamba_dynamicearthnet_semantic.pt'
TRAIN_REPORT = OUTPUT_ROOT / 'dynamicearthnet_semantic_training.json'
CACHE_ROOT = Path('/content/dynamicearthnet_cache')
command = [
    sys.executable, 'scripts/train_dynamicearthnet_semantic_cd.py',
    '--archive', str(ARCHIVE), '--splits', 'data/dynamicearthnet/splits.csv',
    '--cache-dir', str(CACHE_ROOT), '--epochs', '20', '--batch-size', '1',
    '--pairs-per-area', '8', '--validation-pairs-per-area', '8',
    '--tiles-per-pair', '8', '--validation-tiles-per-pair', '8',
    '--device', 'cuda', '--output-checkpoint', str(CHECKPOINT), '--report', str(TRAIN_REPORT),
    '--resume-from', str(CHECKPOINT),
]
subprocess.run(command, check=True)
training_report = json.loads(TRAIN_REPORT.read_text())
print(json.dumps(training_report['validation_metrics'], indent=2))

## Evaluate held-out AOIs
This compares predictions with DynamicEarthNet labels on the AOIs held out during training, and reports the pixel-difference baseline. These are validation results on AOIs from the listed training split, not official test-set metrics. Full evaluation processes all 16 tiles for each selected date pair and can take time.

In [ ]:
locations = ['den' + area_id for area_id in training_report['validation_area_ids']]
EVAL_REPORT = OUTPUT_ROOT / 'dynamicearthnet_semantic_evaluation.json'
command = [
    sys.executable, 'scripts/evaluate_dynamicearthnet_semantic_cd.py',
    '--archive', str(ARCHIVE), '--cache-dir', str(CACHE_ROOT),
    '--checkpoint', str(CHECKPOINT), '--locations', *locations,
    '--device', 'cuda', '--output', str(EVAL_REPORT),
]
subprocess.run(command, check=True)
evaluation_report = json.loads(EVAL_REPORT.read_text())
print(json.dumps({
    'semantic_mamba_binary': evaluation_report['semantic_mamba_binary'],
    'pixel_diff_binary': evaluation_report['pixel_diff_binary'],
    'semantic_transitions': evaluation_report['semantic_transition_metrics'],
}, indent=2))

## Use the checkpoint in TerraEyes
Download `mamba_dynamicearthnet_semantic.pt` from Drive to `models/change_detection_candidates/mamba_dynamicearthnet_semantic.pt`, or set `TERRAEYES_SEMANTIC_MAMBA_CHECKPOINT_PATH` to its location. Keep the detector opt-in with `TERRAEYES_CHANGE_DETECTOR=semantic_mamba` until you review held-out metrics. Training and evaluation JSON reports are saved alongside the checkpoint.